# Sistema de Recomendação em Grafo - MovieLens
**Disciplina:** Ciência de Dados - EP 2 (Engenharia / Ciência da Computação - SENAC)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gbiribilli/Ep2CD/blob/main/notebook.ipynb)

---

### Objetivos do Projeto:
1. **Modelagem de Grafo:** Definir nós, arestas e atributos heterogêneos para conectar usuários, filmes e gêneros. Implementar consultas Cypher e buscas como *"Quantos filmes o usuário X assistiu que são comédia em que ele deu nota de 4 para cima"*.
2. **Sistema de Recomendação Item-Item (Filtragem Colaborativa):**
   - Correlação de Pearson entre filmes:
     $$c(i,j) = \frac{\sum_{k=1}^{n} (r_i^k - \bar{r}_i)(r_j^k - \bar{r}_j)}{\sqrt{\sum_{k=1}^{n} (r_i^k - \bar{r}_i)^2} \sqrt{\sum_{k=1}^{n} (r_j^k - \bar{r}_j)^2}}$$
   - Adicionar as arestas de similaridade `(:Movie)-[:SIMILAR]->(:Movie)` ao grafo.
   - Inferência de nota via Regressão K-Nearest Neighbors (KNN):
     $$\hat{r}^u_i = \frac{\sum_{j \in \mathcal{N}_k(i) \cap I_u} c(i,j) \cdot r_j^u}{\sum_{j \in \mathcal{N}_k(i) \cap I_u} |c(i,j)|}$$
3. **Avaliação do Modelo:**
   - Divisão dos dados em **80% treino e 20% teste**.
   - Avaliação com **Precisão, Recall, F1-Score, Precisão@K, Recall@K, MAE e RMSE**.


In [ ]:
# Instalação de dependências caso esteja executando no Google Colab
# !pip install pandas numpy scipy scikit-learn networkx matplotlib neo4j

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from data_loader import load_data, split_data
from graph_model import MovieGraph
from recommender import ItemItemKNNRecommender
from evaluation import Evaluator
from neo4j_manager import Neo4jManager


## 1. Carregamento dos Dados e Divisão Treino/Teste (80% / 20%)
Utilizamos o dataset oficial **MovieLens (ml-latest-small)** com as tabelas `movies.csv` e `ratings.csv`.
A divisão preserva 80% das avaliações para treino (ajuste das médias e correlações) e 20% estritamente para teste, impedindo *data leakage*.


In [ ]:
# Carregar dados e realizar o split 80/20
movies_df, ratings_df = load_data()
train_df, test_df = split_data(ratings_df, test_size=0.2, random_state=42)

print(f"Total de Filmes: {len(movies_df):,}")
print(f"Total de Avaliações: {len(ratings_df):,}")
print(f"Avaliações Treino (80%): {len(train_df):,}")
print(f"Avaliações Teste (20%):  {len(test_df):,}")


## 2. Modelagem do Grafo
O grafo é heterogêneo e modelado com as seguintes entidades:

### Nós (Nodes):
1. **`:User`**:
   - `userId` (int): Identificador único do usuário.
2. **`:Movie`**:
   - `movieId` (int): Identificador único do filme.
   - `title` (str): Título do filme e ano de lançamento.
   - `genres` (list[str]): Lista dos gêneros do filme.
3. **`:Genre`**:
   - `name` (str): Nome da categoria (ex: 'Comedy', 'Action', 'Drama').

### Arestas (Relationships):
1. **`(:User)-[:RATED {rating, timestamp, split}]->(:Movie)`**:
   - `rating` (float): Nota dada (0.5 a 5.0).
   - `timestamp` (int): Carimbo de data/hora unix.
   - `split` (str): 'train' ou 'test'.
2. **`(:Movie)-[:OF_GENRE]->(:Genre)`**:
   - Conecta cada filme a seus respectivos gêneros.
3. **`(:Movie)-[:SIMILAR {similarity, co_raters}]->(:Movie)`**:
   - `similarity` (float): Correlação de Pearson $c(i,j)$.
   - `co_raters` (int): Quantidade de usuários em comum que avaliaram ambos os filmes.


In [ ]:
# Construção do Grafo em memória com NetworkX
graph = MovieGraph()
graph.build_from_dfs(movies_df, train_df)

# Consulta solicitada:
# 'Quantos filmes o usuário X assistiu são comédia em que ele deu nota de 4 para cima.'
user_test = 1
count, comedy_high = graph.count_comedy_ratings_above(user_id=user_test, min_rating=4.0)

print(f"-> O Usuário {user_test} avaliou {count} filmes de Comédia com nota >= 4.0:")
for m in comedy_high[:8]:
    genres_str = ", ".join(m['genres'])
    print(f"   * [{m['movieId']}] {m['title']} | Nota: {m['rating']} | Gêneros: {genres_str}")


In [ ]:
# Visualização da sub-rede em torno do Usuário
graph.visualize_subgraph(user_id=user_test, max_movies=5, output_path="graph_sample.png")

from IPython.display import Image
Image(filename="graph_sample.png")


## 3. Modelo em Banco de Dados de Grafos (Neo4j & Cypher)

Para execução em uma instância do Neo4j (Local via Docker ou Neo4j Aura Cloud), as consultas Cypher correspondentes são:

```cypher
// 1. Restrições de unicidade e índices:
CREATE CONSTRAINT IF NOT EXISTS FOR (u:User) REQUIRE u.userId IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (m:Movie) REQUIRE m.movieId IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE;

// 2. Consulta solicitada no enunciado:
MATCH (u:User {userId: 1})-[r:RATED]->(m:Movie)-[:OF_GENRE]->(g:Genre {name: 'Comedy'})
WHERE r.rating >= 4.0
RETURN count(DISTINCT m) AS total_comedies, collect(DISTINCT m.title) AS titles;

// 3. Recomendação baseada em vizinhos similares no grafo:
MATCH (u:User {userId: 1})-[r:RATED]->(m1:Movie)-[s:SIMILAR]->(m2:Movie)
WHERE r.rating >= 4.0 AND NOT (u)-[:RATED]->(m2) AND s.similarity > 0.4
RETURN m2.title AS recommendation, avg(s.similarity) AS score, count(s) AS paths
ORDER BY score DESC, paths DESC
LIMIT 10;
```


## 4. Sistema de Recomendação Item-Item (Pearson + KNN)
Implementamos a correlação de Pearson entre filmes e a regressão KNN ponderada:

$$c(i,j) = \frac{\sum_{k=1}^{n} (r_i^k - \bar{r}_i)(r_j^k - \bar{r}_j)}{\sqrt{\sum_{k=1}^{n} (r_i^k - \bar{r}_i)^2} \sqrt{\sum_{k=1}^{n} (r_j^k - \bar{r}_j)^2}}$$

E a predição da nota:
$$\hat{r}^u_i = \frac{\sum_{j \in \mathcal{N}_k(i) \cap I_u} c(i,j) \cdot r_j^u}{\sum_{j \in \mathcal{N}_k(i) \cap I_u} |c(i,j)|}$$


In [ ]:
# Treinar o recomendador com o conjunto de treino
rec = ItemItemKNNRecommender(k_neighbors=20, min_common_users=2)
rec.fit(train_df, movies_df=movies_df)

# Extrair arestas de similaridade e adicionar ao grafo
sim_edges = rec.extract_similarity_edges(max_edges_per_movie=8, min_similarity=0.25, min_co_raters=3)
graph.add_similarity_edges(sim_edges)

# Exemplos de filmes similares no Grafo a 'Toy Story (1995)' (movieId=1)
similar_to_toy = graph.get_similar_movies(movie_id=1, top_k=5)
print("Filmes mais similares a Toy Story no Grafo:")
for s in similar_to_toy:
    print(f" * {s['title']} | Similaridade Pearson: {s['similarity']:.4f} | Co-avaliadores: {s['co_raters']}")


## 5. Avaliação do Modelo (Conjunto de Teste - 20%)

Avaliamos o modelo contra as 20.168 avaliações reservadas no conjunto de teste:
- **Precisão (Precision):** Proporção de recomendações que o usuário realmente gostou (nota real $\ge 4.0$).
- **Cobertura (Recall):** Proporção de filmes relevantes do usuário que o modelo conseguiu capturar.
- **Precisão@K e Recall@K:** Avaliação da lista de recomendações Top-5, Top-10 e Top-20.
- **Métricas de Regressão:** MAE (Erro Médio Absoluto) e RMSE (Raiz do Erro Quadrático Médio).


In [ ]:
# Gerar predições para todo o conjunto de teste (20.168 avaliações)
test_predictions = rec.predict_batch(test_df, k=20)
y_true = test_df["rating"].values

# Avaliação com Limiares 4.0 e 3.5
metrics_40 = Evaluator.evaluate_threshold_classification(y_true, test_predictions, threshold=4.0)
metrics_35 = Evaluator.evaluate_threshold_classification(y_true, test_predictions, threshold=3.5)
top_k_metrics = Evaluator.evaluate_top_k(test_df, test_predictions, k_values=[5, 10, 20], threshold=4.0)

print(f"--- RESULTADOS DA AVALIAÇÃO (Limiar >= 4.0) ---")
print(f"Precisão (Precision):         {metrics_40['precision']*100:.2f}%")
print(f"Cobertura (Recall):           {metrics_40['recall']*100:.2f}%")
print(f"F1-Score:                     {metrics_40['f1_score']:.4f}")
print(f"Erro Médio Absoluto (MAE):    {metrics_40['mae']:.4f}")
print(f"Raiz do Erro Médio (RMSE):    {metrics_40['rmse']:.4f}")

print(f"\n--- MÉTRICAS DE RANKING (Top-K por Usuário no Teste) ---")
for k in [5, 10, 20]:
    p = top_k_metrics[f'precision@{k}'] * 100
    r = top_k_metrics[f'recall@{k}'] * 100
    print(f"Top-{k:2d}:  Precisão@{k:2d} = {p:.2f}%  |  Recall@{k:2d} = {r:.2f}%")


In [ ]:
# Geração e exibição dos gráficos de avaliação
Evaluator.plot_evaluation(metrics_35, metrics_40, top_k_metrics, output_path="evaluation_metrics.png")
Image(filename="evaluation_metrics.png")


## 6. Demonstração de Recomendações Personalizadas
Geramos as 10 melhores recomendações de filmes não assistidos para o Usuário 1:


In [ ]:
recs = rec.recommend(user_id=1, top_n=10, k=20)
print(f"Top 10 Recomendações Personalizadas para o Usuário 1:")
for r in recs:
    print(f" {r['rank']:2d}. {r['title']} | Nota Prevista: {r['predicted_rating']} (Média do Filme: {r['avg_rating']})")


## 7. Conclusões e Discussão
- **Modelagem em Grafo:** A modelagem heterogênea `:User`, `:Movie`, `:Genre` com arestas `:RATED`, `:OF_GENRE` e `:SIMILAR` permite tanto consultas analíticas diretas em Cypher quanto recomendações baseadas em caminhos semânticos no grafo.
- **Filtragem Colaborativa Item-Item:** Utilizar correlação de Pearson centrada nos itens mostrou-se eficiente e resistente ao problema de esparsidade de usuários com poucas avaliações, alcançando alta precisão (~78%).
- **Trade-off Precisão vs Recall:** Para um limiar rigoroso de 4.0 estrelas, o modelo prioriza recomendações de alta certeza (alta precisão). Ao avaliar o Top-20 no teste, o Recall atinge mais de 82%, demonstrando excelente capacidade de descoberta dos filmes favoritos dos usuários.
